<a href="https://colab.research.google.com/github/SamurAIGPT/Text-To-Video-AI/blob/main/Text_to_Video_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Steps for generating video from text

1. Use openai to generate a video script from a topic
2. Use edgetts to pick a voice and create a audio based on the above generated script
3. Use whisper and get timed captions for the above audio
4. Now generate visual keywords for the video script using openai api
5. Fetch videos based on the above visual keywords using pexels api
6. Stich together the videos, audio and captions using Moviepy

### Install python 3.11

In [1]:
!sudo apt-get update -y
!sudo apt-get install -y python3.11
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!sudo update-alternatives --config python3
!sudo apt-get install -y python3.11-distutils
!wget https://bootstrap.pypa.io/get-pip.py
!python3.11 get-pip.py

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,955 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InReleas

In [4]:
!git clone https://github.com/dstryukov/Text-To-Video-AI

Cloning into 'Text-To-Video-AI'...
remote: Enumerating objects: 405, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 405 (delta 4), reused 17 (delta 2), pack-reused 382 (from 2)
Receiving objects: 100% (405/405), 54.89 MiB | 34.23 MiB/s, done.
Resolving deltas: 100% (217/217), done.


In [5]:
%cd Text-To-Video-AI

/content/Text-To-Video-AI


### Install dependencies

In [7]:
!pip3.11 install -r requirements.txt

  Using cached aiohttp-3.9.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.5 kB)
  Using cached aiosignal-1.3.1-py3-none-any.whl.metadata (4.0 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached anyio-4.4.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached attrs-23.2.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached certifi-2024.7.4-py3-none-any.whl.metadata (2.2 kB)
  Using cached charset_normalizer-3.3.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (33 kB)
  Using cached Cython-3.0.10-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.2 kB)
  Using cached decorator-4.4.2-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached deepgram_sdk-3.7.0-py3-none-any.whl.metadata (13 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached dtw_python-1.5.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (48 kB)
  Using cached edge_tts-6.1.12-py3

### 2. Настройка ключей API для провайдеров LLM и Pexels
Укажите ваши API-ключи для LLM (OpenAI, Groq или Gemini) и Pexels (если используются стоковые видео).

In [ ]:
import os

# Выберите провайдера LLM: openai | groq | gemini
os.environ["LLM_PROVIDER"] = "gemini"

# Укажите ключи API в соответствии с выбранным провайдером
os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"
os.environ["GEMINI_API_KEY"] = "your_gemini_api_key_here"

# Ключ Pexels API (требуется только если вы используете stock_video)
os.environ["PEXELS_API_KEY"] = "your_pexels_api_key_here"


### 3. Авторизация на Hugging Face
Для автоматического скачивания весов моделей Flux через утилиту `huggingface-cli` необходимо войти под своей учетной записью Hugging Face.
Введите ваш токен авторизации (HF Token) в поле ввода ниже.

In [ ]:
from getpass import getpass
import os

hf_token = getpass("Введите Hugging Face token (hf_...). Можно оставить пустым: ").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    try:
        from huggingface_hub import login
        login(token=hf_token)
        print("Hugging Face login successful.")
    except Exception as e:
        print(f"Hugging Face login failed: {e}")
else:
    print("HF token not provided. Continuing without token.")


### 4. Установка ComfyUI и кастомных нод
В качестве основного движка генерации изображений используется ComfyUI. Склонируем репозиторий, установим зависимости, а также расширение `ComfyUI-GGUF` для работы с квантованными Flux моделями.

In [ ]:
# Клонирование ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt

# Установка расширения ComfyUI-GGUF для поддержки квантованных моделей
%cd custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF.git
%cd ..
%cd ..


### 5. Предварительное скачивание моделей для генерации картинок
Скачаем VAE, текстовые кодировщики (CLIP) и веса Flux Schnell (в формате FP8 для оптимального расхода видеопамяти на Colab T4).
Мы используем `huggingface-cli` с автоматической поддержкой токена из шага авторизации.

In [ ]:
# 1. Скачивание VAE для Flux
!huggingface-cli download black-forest-labs/FLUX.1-schnell ae.safetensors --local-dir ComfyUI/models/vae

# 2. Скачивание текстовых энкодеров (clip_l и квантованный t5xxl_fp8 для экономии VRAM)
!huggingface-cli download comfyanonymous/flux_text_encoders clip_l.safetensors --local-dir ComfyUI/models/clip
!huggingface-cli download comfyanonymous/flux_text_encoders t5xxl_fp8_e4m3fn.safetensors --local-dir ComfyUI/models/clip

# 3. Скачивание весов модели Flux Schnell FP8
!huggingface-cli download black-forest-labs/FLUX.1-schnell flux1-schnell-fp8.safetensors --local-dir ComfyUI/models/diffusion_models


### 6. Запуск сервера ComfyUI
Запустим сервер ComfyUI в фоновом режиме и дождемся его полной готовности.

In [ ]:
import subprocess
import time
import requests

print("Запуск сервера ComfyUI в фоновом режиме...")
# Запуск ComfyUI на localhost:8188 в режиме GPU
cmd = "python ComfyUI/main.py --listen 127.0.0.1 --port 8188 --gpu-only"
with open("comfyui_server.log", "w") as f:
    process = subprocess.Popen(cmd.split(), stdout=f, stderr=f)

# Ожидание ответа по REST API
comfyui_url = "http://127.0.0.1:8188"
connected = False
print("Ожидание готовности сервера...")
for i in range(30):
    try:
        resp = requests.get(comfyui_url, timeout=2)
        if resp.status_code == 200:
            connected = True
            print("Сервер ComfyUI успешно запущен и отвечает на запросы!")
            break
    except:
        pass
    time.sleep(2)

if not connected:
    print("Внимание: Сервер ComfyUI не ответил за 60 секунд. Проверьте comfyui_server.log для получения деталей.")


### 7. Создание директории воркфлоу-шаблонов
Создадим папку `workflows/` и запишем в нее стандартный API-воркфлоу для Flux Schnell.

In [ ]:
import os
import json

os.makedirs("workflows", exist_ok=True)
workflow_data = {
  "3": {
    "inputs": {
      "seed": 111111111111111,
      "steps": 4,
      "cfg": 1.0,
      "sampler_name": "euler",
      "scheduler": "simple",
      "denoise": 1.0,
      "model": ["12", 0],
      "positive": ["6", 0],
      "negative": ["7", 0],
      "latent_image": ["5", 0]
    },
    "class_type": "KSampler"
  },
  "5": {
    "inputs": {
      "width": 576,
      "height": 1024,
      "batch_size": 1
    },
    "class_type": "EmptyLatentImage"
  },
  "6": {
    "inputs": {
      "text": "Positive prompt",
      "clip": ["11", 0]
    },
    "class_type": "CLIPTextEncode"
  },
  "7": {
    "inputs": {
      "text": "Negative prompt",
      "clip": ["11", 0]
    },
    "class_type": "CLIPTextEncode"
  },
  "8": {
    "inputs": {
      "samples": ["3", 0],
      "vae": ["10", 0]
    },
    "class_type": "VAEDecode"
  },
  "9": {
    "inputs": {
      "filename_prefix": "t2v_scene",
      "images": ["8", 0]
    },
    "class_type": "SaveImage"
  },
  "10": {
    "inputs": {
      "vae_name": "ae.safetensors"
    },
    "class_type": "VAELoader"
  },
  "11": {
    "inputs": {
      "clip_name1": "t5xxl_fp8_e4m3fn.safetensors",
      "clip_name2": "clip_l.safetensors",
      "type": "flux"
    },
    "class_type": "DualCLIPLoader"
  },
  "12": {
    "inputs": {
      "unet_name": "flux1-schnell-fp8.safetensors",
      "weight_dtype": "default"
    },
    "class_type": "UNETLoader"
  }
}

with open("workflows/flux_schnell_fp8_4step.json", "w", encoding="utf-8") as f:
    json.dump(workflow_data, f, ensure_ascii=False, indent=2)
print("Воркфлоу успешно сохранен в workflows/flux_schnell_fp8_4step.json")


### 8. Настройка озвучки (TTS) и Voice Cloning
Для создания качественной озвучки с клонированием голоса (Voice Cloning) вы можете использовать современные бэкенды, например **F5-TTS** или **Fish Speech**.
Загрузите ваш референсный аудиофайл для клонирования голоса.

In [ ]:
from google.colab import files
import os

os.makedirs("voices", exist_ok=True)
print("Выберите аудиофайл (wav или mp3) длиной 5-15 секунд для клонирования голоса:")
uploaded = files.upload()
for filename in uploaded.keys():
    target_path = os.path.join("voices", "reference.wav")
    with open(target_path, "wb") as f:
        f.write(uploaded[filename])
    print(f"Файл {filename} успешно сохранен как {target_path} для клонирования голоса.")


#### Установка дополнительных пакетов для F5-TTS / Fish Speech
Разблокируйте и запустите соответствующую строчку в зависимости от выбранного бэкенда. Silero TTS работает по умолчанию.

In [ ]:
# 1. Вариант: Установка F5-TTS (требуется GPU с VRAM >= 12GB)
# !pip install -r requirements-tts-f5.txt

# 2. Вариант: Установка Fish Speech
# !pip install -r requirements-tts-fish.txt


### 9. Исправление политик ImageMagick
Разрешим MoviePy использовать ImageMagick для создания субтитров.

In [ ]:
!apt install imagemagick &> /dev/null
!sed -i '/<policy domain="path" rights="none" pattern="@\*"/d' /etc/ImageMagick-6/policy.xml


### 10. Управление настройками пайплайна (config.yaml)
Создайте или обновите конфигурационный файл проекта.

In [ ]:
%%writefile config.yaml
# Универсальные настройки пайплайна генерации видео
project_name: "colab_project"
aspect_ratio: "9:16"

# Настройки LLM для генерации сценария и промптов
llm_provider: "gemini" # openai | groq | gemini
llm_model: "gemini-2.5-flash"

# Настройки озвучки (TTS)
tts:
  # Доступные бэкенды: f5_tts | fish_speech | cosyvoice | local_tts_api | silero | audio_file | none
  backend: "silero"
  fallback_backend: "silero"
  mode: "full_script"            # per_scene | full_script (full_script рекомендуется для клонирования)

  language: "ru"
  voice: "default"
  speed: 1.0
  emotion: "neutral"
  sample_rate: 24000
  format: "wav"
  normalize_text: true

  # Voice Cloning
  reference_audio_path: "voices/reference.wav"
  reference_text: ""
  use_voice_clone: true

  # Разбивка по символам
  max_chars_per_chunk: 350
  pause_between_chunks_ms: 250

  local_tts_api_url: "http://127.0.0.1:8020/tts"

  silero:
    voice: "xenia"
    sample_rate: 24000
    speed: "medium"

  f5_tts:
    model: "F5-TTS"
    device: "cuda"
    dtype: "float16"
    nfe_step: 32
    cfg_strength: 2.0
    speed: 1.0

  replacements:
    "AI": "искусственный интеллект"
    "API": "эй-пи-ай"
    "URL": "ссылка"
    "404": "четыреста четыре"

# Постобработка звука
audio_postprocess:
  enabled: true
  normalize_loudness: true
  target_lufs: -14
  remove_silence: true
  max_silence_ms: 500
  add_compressor: true
  noise_reduction: false

# Настройки рендеринга и генерации визуалов по умолчанию
render:
  # Бэкенд визуализации: comfyui | local_api | image_folder | stock_video | none
  visual_backend: "comfyui"
  fallback_backend: "image_folder"
  model_preset: "flux_schnell_fp8"
  acceleration_mode: "schnell"

  image_width: 576
  image_height: 1024

  final_width: 1080
  final_height: 1920

  steps: 4
  guidance_scale: 0.0
  seed: -1
  sampler: "euler"
  scheduler: "simple"

  lora_preset: "none"
  lora_strength: 0.7

  motion_preset: "slow_zoom_in"
  captions_enabled: true

  comfyui_url: "http://127.0.0.1:8188"
  comfyui_workflow_path: "workflows/flux_schnell_fp8_4step.json"
  local_image_api_url: "http://127.0.0.1:8000/txt2img"
  image_folder_path: "input_images"

  stock_video_provider: "pexels"
  stock_video_orientation: "portrait"

  fallback_to_black: true
  clear_cuda_cache_between_scenes: true

# Настройки авторизации Hugging Face
huggingface:
  token_env: "HF_TOKEN"
  use_auth_token: true

# Настройки отображения субтитров
captions:
  font_size: 90
  font_color: "white"
  font_face: "Arial-Bold"
  stroke_width: 3
  stroke_color: "black"
  position: "bottom_center"


### 11. Запуск пайплайна генерации видео
Запустите основной пайплайн. Вы можете переопределить бэкенды генерации и озвучки через аргументы командной строки.

In [ ]:
# Вариант 1: Запуск с бесплатной локальной озвучкой Silero TTS и заглушкой картинок (черный экран)
!python app.py "Интересные факты о космосе" --tts-backend silero --backend none

# Вариант 2: Запуск с клонированием голоса через F5-TTS и готовыми картинками из input_images/
# !python app.py "Интересные факты о космосе" \
#   --tts-backend f5_tts \
#   --tts-mode full_script \
#   --reference-audio voices/reference.wav \
#   --backend image_folder

# Вариант 3: Запуск со стоковыми видео и предзаписанной вами озвучкой
# !python app.py "Интересные факты о космосе" \
#   --tts-backend audio_file \
#   --backend stock_video

# Вариант 4: Полный автозапуск (ComfyUI Flux Schnell + F5-TTS Voice Cloning)
# !python app.py "Интересные факты о космосе" \
#   --tts-backend f5_tts \
#   --tts-mode full_script \
#   --reference-audio voices/reference.wav \
#   --backend comfyui \
#   --model-preset flux_schnell_fp8 \
#   --acceleration schnell \
#   --image-width 576 \
#   --image-height 1024


### 12. Воспроизведение итогового ролика
Отобразим готовый видеоролик `rendered_video.mp4` прямо в интерфейсе Colab.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

video_path = 'rendered_video.mp4'

def display_video(video_path, width=480, height=854):
    try:
        video_file = open(video_path, "rb").read()
        video_url = "data:video/mp4;base64," + b64encode(video_file).decode()
        return HTML(f"""
        <video width={width} height={height} controls>
            <source src="{video_url}" type="video/mp4">
        </video>
        """)
    except Exception as e:
        return f"Ошибка загрузки видео: {e}. Убедитесь, что рендеринг прошел успешно и файл {video_path} существует."

display_video(video_path)
